In [2]:
import subprocess
subprocess.run(["pip", "install", "hdfs", "--quiet"], check=True)
print("✅ Pronto")

✅ Pronto


In [4]:
from pyspark.sql import SparkSession
# Encerra sessão anterior se existir
try:
    spark.stop()
except:
    pass

spark = SparkSession.builder \
    .appName("citibike-bigdata") \
    .config("spark.hadoop.dfs.client.use.datanode.hostname", "true") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://localhost:9000") \
    .getOrCreate()

print(spark.sparkContext.getConf().get("spark.hadoop.dfs.client.use.datanode.hostname"))
# Deve imprimir: true

true


In [5]:
# Verifica o que chegou no HDFS
from hdfs import InsecureClient
hdfs_client = InsecureClient("http://localhost:9870", user="root")

# Lista arquivos em /citibike/trips/
arquivos = hdfs_client.list("/citibike/trips/", status=True)
print(f"📁 {len(arquivos)} arquivo(s) em /citibike/trips/\n")
for nome, info in sorted(arquivos):
    tamanho_mb = info['length'] / 1024 / 1024
    print(f"  {nome:55s} {tamanho_mb:7.1f} MB")

📁 11 arquivo(s) em /citibike/trips/

  202601-citibike-tripdata_1.csv                            185.8 MB
  202601-citibike-tripdata_2.csv                            151.6 MB
  202602-citibike-tripdata_1.csv                            185.8 MB
  202602-citibike-tripdata_2.csv                             40.8 MB
  202603-citibike-tripdata_1.csv                            186.2 MB
  202603-citibike-tripdata_2.csv                            185.7 MB
  202603-citibike-tripdata_3.csv                            176.3 MB
  202604-citibike-tripdata-part1.csv                        180.0 MB
  202604-citibike-tripdata-part2.csv                        179.6 MB
  202604-citibike-tripdata-part3.csv                        179.3 MB
  202604-citibike-tripdata-part4.csv                        179.8 MB


In [6]:
from hdfs import InsecureClient

hdfs = InsecureClient("http://localhost:9870", user="root")
arquivos = hdfs.list("/citibike/trips/")
print(f"📁 {len(arquivos)} arquivo(s):\n")
for f in sorted(arquivos):
    info = hdfs.status(f"/citibike/trips/{f}")
    print(f"  {f:55s} {info['length']/1024/1024:7.1f} MB")

📁 11 arquivo(s):

  202601-citibike-tripdata_1.csv                            185.8 MB
  202601-citibike-tripdata_2.csv                            151.6 MB
  202602-citibike-tripdata_1.csv                            185.8 MB
  202602-citibike-tripdata_2.csv                             40.8 MB
  202603-citibike-tripdata_1.csv                            186.2 MB
  202603-citibike-tripdata_2.csv                            185.7 MB
  202603-citibike-tripdata_3.csv                            176.3 MB
  202604-citibike-tripdata-part1.csv                        180.0 MB
  202604-citibike-tripdata-part2.csv                        179.6 MB
  202604-citibike-tripdata-part3.csv                        179.3 MB
  202604-citibike-tripdata-part4.csv                        179.8 MB


In [7]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("hdfs://localhost:9000/citibike/trips/202601-citibike-tripdata_1.csv")

print(f"✅ {df.count():,} registros")
print(f"Colunas: {df.columns}")
df.printSchema()

✅ 1,000,000 registros
Colunas: ['ride_id', 'rideable_type', 'started_at', 'ended_at', 'start_station_name', 'start_station_id', 'end_station_name', 'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng', 'member_casual']
root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: timestamp (nullable = true)
 |-- ended_at: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- start_lat: double (nullable = true)
 |-- start_lng: double (nullable = true)
 |-- end_lat: double (nullable = true)
 |-- end_lng: double (nullable = true)
 |-- member_casual: string (nullable = true)



In [9]:
from pyspark.sql.functions import to_timestamp, col
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# 1. Cria coluna de duração
df_dur = df \
    .withColumn("started_at", to_timestamp("started_at")) \
    .withColumn("ended_at",   to_timestamp("ended_at")) \
    .withColumn("duration_min",
        (col("ended_at").cast("long") - col("started_at").cast("long")) / 60) \
    .filter((col("duration_min") > 0) & (col("duration_min") < 60))  # cap em 60min

# 2. Estatísticas descritivas
print("📊 Duração das viagens (minutos):")
df_dur.select("duration_min").describe().show()

📊 Duração das viagens (minutos):
+-------+------------------+
|summary|      duration_min|
+-------+------------------+
|  count|            995615|
|   mean|  9.85254378784297|
| stddev|7.8239350348586045|
|    min|1.0166666666666666|
|    max|59.983333333333334|
+-------+------------------+



In [10]:
import subprocess
subprocess.run(["pip", "install", "plotly", "--quiet"], check=True)
import plotly.io as pio
pio.renderers.default = "iframe"  
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Coleta amostra para o Plotly
df_tipo = df_dur.select("duration_min", "member_casual") \
    .sample(fraction=0.1, seed=42).toPandas()

members = df_tipo[df_tipo["member_casual"] == "member"]["duration_min"]
casuals = df_tipo[df_tipo["member_casual"] == "casual"]["duration_min"]

# Subplots: geral + por tipo
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Distribuição Geral", "Por Tipo de Usuário")
)

# Histograma geral
fig.add_trace(
    go.Histogram(x=df_tipo["duration_min"], nbinsx=50,
                 marker_color="#4f8ef7", name="Geral",
                 hovertemplate="Duração: %{x:.1f} min<br>Viagens: %{y}<extra></extra>"),
    row=1, col=1
)

# Por tipo
fig.add_trace(
    go.Histogram(x=members, nbinsx=50, name="Member",
                 marker_color="#34c97a", opacity=0.75,
                 hovertemplate="Duração: %{x:.1f} min<br>Viagens: %{y}<extra></extra>"),
    row=1, col=2
)
fig.add_trace(
    go.Histogram(x=casuals, nbinsx=50, name="Casual",
                 marker_color="#f5a623", opacity=0.75,
                 hovertemplate="Duração: %{x:.1f} min<br>Viagens: %{y}<extra></extra>"),
    row=1, col=2
)

fig.update_layout(
    title_text="Duração das Viagens CitiBike — Janeiro 2026",
    title_font_size=16,
    barmode="overlay",
    plot_bgcolor="#1c1b19",
    paper_bgcolor="#171614",
    font_color="#cdccca",
    legend=dict(bgcolor="#1c1b19", bordercolor="#393836", borderwidth=1),
    height=450
)

fig.update_xaxes(title_text="Minutos", gridcolor="#262523", zeroline=False)
fig.update_yaxes(title_text="Nº de viagens", gridcolor="#262523", zeroline=False)

fig.show()

In [11]:
import plotly.express as px

fig2 = px.box(
    df_tipo, x="member_casual", y="duration_min",
    color="member_casual",
    color_discrete_map={"member": "#34c97a", "casual": "#f5a623"},
    labels={"member_casual": "Tipo", "duration_min": "Duração (min)"},
    title="Distribuição de Duração por Tipo de Usuário"
)

fig2.update_layout(
    plot_bgcolor="#1c1b19",
    paper_bgcolor="#171614",
    font_color="#cdccca",
    showlegend=False,
    height=400
)
fig2.update_yaxes(gridcolor="#262523")

fig2.show()

In [12]:
from pyspark.sql.functions import col, radians, sin, cos, sqrt, atan2, round as spark_round
import pyspark.sql.functions as F

# Haversine via funções nativas do Spark (sem UDF — mais rápido)
def add_distance_km(df):
    R = 6371  # raio da Terra em km
    return df \
        .withColumn("lat1", radians(col("start_lat"))) \
        .withColumn("lat2", radians(col("end_lat"))) \
        .withColumn("dlat", radians(col("end_lat") - col("start_lat"))) \
        .withColumn("dlon", radians(col("end_lng") - col("start_lng"))) \
        .withColumn("a",
            sin(col("dlat")/2)**2 +
            cos(col("lat1")) * cos(col("lat2")) * sin(col("dlon")/2)**2
        ) \
        .withColumn("distance_km",
            spark_round(2 * R * atan2(sqrt(col("a")), sqrt(1 - col("a"))), 3)
        ) \
        .drop("lat1", "lat2", "dlat", "dlon", "a") \
        .filter((col("distance_km") > 0.05) & (col("distance_km") < 30))

df_geo = add_distance_km(df)
print(f"✅ {df_geo.count():,} viagens com trajeto válido")

✅ 981,680 viagens com trajeto válido


In [13]:
print("📍 Distância dos trajetos (km):")
df_geo.select("distance_km").describe().show()

# Por tipo de usuário
df_geo.groupBy("member_casual").agg(
    spark_round(F.avg("distance_km"), 3).alias("média km"),
    spark_round(F.percentile_approx("distance_km", 0.5), 3).alias("mediana km"),
    spark_round(F.max("distance_km"), 3).alias("máx km"),
    F.count("*").alias("viagens")
).orderBy("média km", ascending=False).show()

📍 Distância dos trajetos (km):
+-------+------------------+
|summary|       distance_km|
+-------+------------------+
|  count|            981680|
|   mean| 1.820285457582893|
| stddev|1.4868410487605943|
|    min|             0.058|
|    max|            26.462|
+-------+------------------+

+-------------+--------+----------+------+-------+
|member_casual|média km|mediana km|máx km|viagens|
+-------------+--------+----------+------+-------+
|       casual|   2.015|     1.579|19.817|  93800|
|       member|     1.8|     1.358|26.462| 887880|
+-------------+--------+----------+------+-------+



In [14]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "iframe"

# Amostra para o plot
sample = df_geo.select("distance_km", "member_casual") \
    .sample(fraction=0.1, seed=42).toPandas()

members = sample[sample["member_casual"] == "member"]["distance_km"]
casuals = sample[sample["member_casual"] == "casual"]["distance_km"]

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Distribuição Geral", "Member vs Casual")
)

# Geral
fig.add_trace(
    go.Histogram(x=sample["distance_km"], nbinsx=60,
                 marker_color="#4f8ef7", name="Geral",
                 hovertemplate="Distância: %{x:.2f} km<br>Viagens: %{y}<extra></extra>"),
    row=1, col=1
)

# Por tipo
for nome, dados, cor in [("Member", members, "#34c97a"), ("Casual", casuals, "#f5a623")]:
    fig.add_trace(
        go.Histogram(x=dados, nbinsx=60, name=nome,
                     marker_color=cor, opacity=0.72,
                     hovertemplate="Distância: %{x:.2f} km<br>Viagens: %{y}<extra></extra>"),
        row=1, col=2
    )

fig.update_layout(
    title_text="Distribuição da Kilometragem dos Trajetos — CitiBike Jan 2026",
    title_font_size=15,
    barmode="overlay",
    plot_bgcolor="#1c1b19",
    paper_bgcolor="#171614",
    font_color="#cdccca",
    legend=dict(bgcolor="#1c1b19", bordercolor="#393836", borderwidth=1),
    height=450
)
fig.update_xaxes(title_text="Distância (km)", gridcolor="#262523", dtick=1)
fig.update_yaxes(title_text="Nº de viagens", gridcolor="#262523")
fig.show()

In [15]:
import plotly.graph_objects as go
import plotly.io as pio
import pandas as pd
import numpy as np

pio.renderers.default = "iframe"

# Agrega estações de origem únicas
origens = rotas.groupby("start_station_name").agg(
    lat=("start_lat", "first"),
    lon=("start_lng", "first"),
    total_viagens=("viagens", "sum")
).reset_index()

fig = go.Figure()

# ── Linhas de rota (uma por par origem→destino) ──────────────────────────────
for _, row in rotas.iterrows():
    fig.add_trace(go.Scattermap(
        lat=[row["start_lat"], row["end_lat"], None],
        lon=[row["start_lng"], row["end_lng"], None],
        mode="lines",
        line=dict(width=max(0.5, row["viagens"] / rotas["viagens"].max() * 4),
                  color="#4f8ef7"),
        opacity=0.25,
        hoverinfo="skip",
        showlegend=False,
        name=row["start_station_name"]
    ))

# ── Pontos das estações de origem ─────────────────────────────────────────────
fig.add_trace(go.Scattermap(
    lat=origens["lat"],
    lon=origens["lon"],
    mode="markers",
    marker=dict(
        size=np.clip(origens["total_viagens"] / origens["total_viagens"].max() * 20, 6, 20),
        color=origens["total_viagens"],
        colorscale=[[0, "#4f8ef7"], [0.5, "#f5a623"], [1, "#34c97a"]],
        showscale=True,
        colorbar=dict(title="Viagens", thickness=12,
                      bgcolor="#1c1b19", tickfont=dict(color="#cdccca"))
    ),
    text=origens["start_station_name"],
    customdata=origens["total_viagens"],
    hovertemplate="<b>%{text}</b><br>Viagens saindo: %{customdata:,}<extra></extra>",
    name="Estações"
))

fig.update_layout(
    title=dict(text="Rotas CitiBike NYC — Top 300 pares mais frequentes (Jan 2026)",
               font=dict(size=14, color="#cdccca")),
    map=dict(
        style="carto-darkmatter",
        center=dict(lat=40.73, lon=-73.99),
        zoom=12
    ),
    paper_bgcolor="#171614",
    font_color="#cdccca",
    margin=dict(l=0, r=0, t=40, b=0),
    height=620,
    showlegend=False
)

fig.show()

NameError: name 'rotas' is not defined